# Calidad de datos

In [1]:
# Importacion de librerias

import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow as pa

DB_PATH = '../data/interim/master_events.db'
conn = sqlite3.connect(DB_PATH)

In [2]:
count_query = """
SELECT COUNT(*) AS row_count
FROM master_events;
"""

row_count = pd.read_sql_query(count_query, conn)

row_count

,row_count
0,2095076


In [3]:
schema_query = """
PRAGMA table_info('master_events');
"""

schema = pd.read_sql_query(schema_query, conn)

schema

,cid,name,type,notnull,dflt_value,pk
0,0,index,INT,0,None,0
1,1,event_time,TEXT,0,None,0
2,2,event_type,TEXT,0,None,0
3,3,product_id,INT,0,None,0
4,4,category_id,INT,0,None,0
5,5,category_code,TEXT,0,None,0
6,6,brand,TEXT,0,None,0
7,7,price,REAL,0,None,0
8,8,user_id,INT,0,None,0
9,9,user_session,TEXT,0,None,0


In [4]:
missing_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(event_time) AS event_time_non_null,
    COUNT(event_type) AS event_type_non_null,
    COUNT(product_id) AS product_id_non_null,
    COUNT(category_id) AS category_id_non_null,
    COUNT(category_code) AS category_code_non_null,
    COUNT(brand) AS brand_non_null,
    COUNT(price) AS price_non_null,
    COUNT(user_id) AS user_id_non_null,
    COUNT(user_session) AS user_session_non_null
FROM master_events;
"""

missing = pd.read_sql_query(missing_query, conn)

missing

,total_rows,event_time_non_null,event_type_non_null,product_id_non_null,category_id_non_null,category_code_non_null,brand_non_null,price_non_null,user_id_non_null,user_session_non_null
0,2095076,2095076,2095076,2095076,2095076,34665,1203430,2095076,2095076,2094570


In [5]:
missing_summary = pd.DataFrame({
    "column": schema["name"].iloc[1:].reset_index(drop=True),
    "non_null": [
        missing.loc[0, "event_time_non_null"],
        missing.loc[0, "event_type_non_null"],
        missing.loc[0, "product_id_non_null"],
        missing.loc[0, "category_id_non_null"],
        missing.loc[0, "category_code_non_null"],
        missing.loc[0, "brand_non_null"],
        missing.loc[0, "price_non_null"],
        missing.loc[0, "user_id_non_null"],
        missing.loc[0, "user_session_non_null"],
    ]
})

missing_summary["missing"] = (
    row_count.loc[0, "row_count"] - missing_summary["non_null"]
)

missing_summary["missing_pct"] = (
    missing_summary["missing"] / row_count.loc[0, "row_count"] * 100
)

missing_summary

missing_summary["missing"] = row_count.loc[0, "row_count"] - missing_summary["non_null"]

missing_summary["missing_pct"] = (
    missing_summary["missing"] /
    row_count.loc[0, "row_count"] * 100
)

missing_summary

,column,non_null,missing,missing_pct
0,event_time,2095076,0,0.000000
1,event_type,2095076,0,0.000000
2,product_id,2095076,0,0.000000
3,category_id,2095076,0,0.000000
4,category_code,34665,2060411,98.345406
5,brand,1203430,891646,42.559124
6,price,2095076,0,0.000000
7,user_id,2095076,0,0.000000
8,user_session,2094570,506,0.024152


In [6]:
missing_by_event_query = """
SELECT
    event_type,
    COUNT(*) AS total_events,
    SUM(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END) AS category_code_missing,
    SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END) AS brand_missing,
    SUM(CASE WHEN user_session IS NULL THEN 1 ELSE 0 END) AS session_missing
FROM master_events
GROUP BY event_type
ORDER BY total_events DESC;
"""

missing_by_event = pd.read_sql_query(
    missing_by_event_query,
    conn
)

missing_by_event

,event_type,total_events,category_code_missing,brand_missing,session_missing
0,view,965893,943487,396371,4
1,cart,585854,579073,257165,417
2,remove_from_cart,415754,411950,184144,85
3,purchase,127575,125901,53966,0


In [7]:
missing_by_event["category_code_missing_pct"] = (
    missing_by_event["category_code_missing"]
    / missing_by_event["total_events"] * 100
)

missing_by_event["brand_missing_pct"] = (
    missing_by_event["brand_missing"]
    / missing_by_event["total_events"] * 100
)

missing_by_event["session_missing_pct"] = (
    missing_by_event["session_missing"]
    / missing_by_event["total_events"] * 100
)

missing_by_event

,event_type,total_events,category_code_missing,brand_missing,session_missing,category_code_missing_pct,brand_missing_pct,session_missing_pct
0,view,965893,943487,396371,4,97.680281,41.036740,0.000414
1,cart,585854,579073,257165,417,98.842544,43.895749,0.071178
2,remove_from_cart,415754,411950,184144,85,99.085036,44.291576,0.020445
3,purchase,127575,125901,53966,0,98.687831,42.301391,0.000000


In [8]:
category_product_query = """
SELECT
    product_id,
    COUNT(*) AS event_count,
    COUNT(category_code) AS category_code_non_null,
    SUM(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END) AS category_code_missing
FROM master_events
GROUP BY product_id
ORDER BY category_code_missing DESC;
"""

category_product = pd.read_sql_query(
    category_product_query,
    conn
)

category_product.head(20)

,product_id,event_count,category_code_non_null,category_code_missing
0,5809910,14021,0,14021
1,5809912,5606,0,5606
2,5700037,4838,0,4838
3,5854897,4387,0,4387
4,5751422,4290,0,4290
5,5802432,4263,0,4263
6,5751383,4225,0,4225
7,5849033,4016,0,4016
8,5815662,3887,0,3887
9,5809911,3549,0,3549


In [9]:
brand_product_query = """
SELECT
    product_id,
    COUNT(*) AS event_count,
    COUNT(brand) AS brand_non_null,
    SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END) AS brand_missing
FROM master_events
GROUP BY product_id
ORDER BY brand_missing DESC;
"""

brand_product = pd.read_sql_query(
    brand_product_query,
    conn
)

brand_product.head(20)

,product_id,event_count,brand_non_null,brand_missing
0,5802432,4263,0,4263
1,5815662,3887,0,3887
2,5792800,3238,0,3238
3,5686925,2754,0,2754
4,5528035,2468,0,2468
5,5886282,2295,0,2295
6,5790563,2285,0,2285
7,5789668,1898,0,1898
8,5892179,1841,0,1841
9,5804820,1840,0,1840


In [10]:
category_quality_query = """
SELECT
    product_id,
    COUNT(*) AS event_count,
    COUNT(category_code) AS non_null,
    SUM(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END) AS missing
FROM master_events
GROUP BY product_id;
"""

category_quality = pd.read_sql_query(
    category_quality_query,
    conn
)

category_quality["status"] = np.select(
    [
        category_quality["missing"] == 0,
        category_quality["non_null"] == 0
    ],
    [
        "complete",
        "completely_missing"
    ],
    default="partial"
)

category_quality["status"].value_counts()

status
completely_missing    45527
complete                507
partial                   4
Name: count, dtype: int64

In [11]:
brand_quality_query = """
SELECT
    product_id,
    COUNT(*) AS event_count,
    COUNT(brand) AS non_null,
    SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END) AS missing
FROM master_events
GROUP BY product_id;
"""

brand_quality = pd.read_sql_query(
    brand_quality_query,
    conn
)

brand_quality["status"] = np.select(
    [
        brand_quality["missing"] == 0,
        brand_quality["non_null"] == 0
    ],
    [
        "complete",
        "completely_missing"
    ],
    default="partial"
)

brand_quality["status"].value_counts()

status
complete              22169
completely_missing    20148
partial                3721
Name: count, dtype: int64

# Cargar los datos a pandas

In [12]:
query = """
SELECT *
FROM master_events;
"""

df = pd.read_sql_query(query, conn)

In [13]:
df.shape

(2095076, 10)

In [14]:
df.head()

,index,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,68,2019-10-01 00:01:46 UTC,view,5843665,1487580005092295511,NaN,f.o.x,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361
1,72,2019-10-01 00:01:55 UTC,cart,5868461,1487580013069861041,NaN,italwax,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1
2,95,2019-10-01 00:02:50 UTC,view,5877456,1487580006300255120,NaN,jessnail,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770
3,122,2019-10-01 00:03:41 UTC,view,5649270,1487580013749338323,NaN,concept,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df
4,124,2019-10-01 00:03:44 UTC,view,18082,1487580005411062629,NaN,cnd,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5


In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2095076 entries, 0 to 2095075
Data columns (total 10 columns):
 #   Column         Dtype  
---  ------         -----  
 0   index          int64  
 1   event_time     str    
 2   event_type     str    
 3   product_id     int64  
 4   category_id    int64  
 5   category_code  str    
 6   brand          str    
 7   price          float64
 8   user_id        int64  
 9   user_session   str    
dtypes: float64(1), int64(4), str(5)
memory usage: 299.7 MB


In [16]:
df["event_time"].isna().sum()

np.int64(0)

In [17]:
df["event_time"] = pd.to_datetime(
    df["event_time"],
    utc=True
)

In [18]:
df.head()

,index,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,68,2019-10-01 00:01:46+00:00,view,5843665,1487580005092295511,NaN,f.o.x,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361
1,72,2019-10-01 00:01:55+00:00,cart,5868461,1487580013069861041,NaN,italwax,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1
2,95,2019-10-01 00:02:50+00:00,view,5877456,1487580006300255120,NaN,jessnail,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770
3,122,2019-10-01 00:03:41+00:00,view,5649270,1487580013749338323,NaN,concept,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df
4,124,2019-10-01 00:03:44+00:00,view,18082,1487580005411062629,NaN,cnd,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5


In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2095076 entries, 0 to 2095075
Data columns (total 10 columns):
 #   Column         Dtype              
---  ------         -----              
 0   index          int64              
 1   event_time     datetime64[us, UTC]
 2   event_type     str                
 3   product_id     int64              
 4   category_id    int64              
 5   category_code  str                
 6   brand          str                
 7   price          float64            
 8   user_id        int64              
 9   user_session   str                
dtypes: datetime64[us, UTC](1), float64(1), int64(4), str(4)
memory usage: 253.8 MB


In [20]:
missing_summary = pd.DataFrame({
    "missing": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100
})

missing_summary.sort_values("missing_pct", ascending=False)

,missing,missing_pct
category_code,2060411,98.345406
brand,891646,42.559124
user_session,506,0.024152
index,0,0.000000
event_time,0,0.000000
event_type,0,0.000000
category_id,0,0.000000
product_id,0,0.000000
price,0,0.000000
user_id,0,0.000000


In [21]:
df.duplicated().sum()

np.int64(0)

In [22]:
cols_without_index = [col for col in df.columns if col != "index"]

df.duplicated(subset=cols_without_index).sum()

np.int64(122097)

In [23]:
df["index"].nunique()

1731093

In [24]:
df["index"].duplicated().sum()

np.int64(363983)

In [25]:
len(df), df["index"].nunique()

(2095076, 1731093)

In [26]:
df.duplicated().sum()

cols_without_index = [col for col in df.columns if col != "index"]
df.duplicated(subset=cols_without_index).sum()

df["index"].nunique()

df["index"].duplicated().sum()

np.int64(363983)

In [27]:
df.groupby("index").size().sort_values(ascending=False).head(10)

index
908806     5
836891     5
271053     5
539680     5
1760200    5
887065     5
1147621    5
1476336    5
1823368    5
2003207    5
dtype: int64

In [28]:
df["month"] = df["event_time"].dt.to_period("M")

C:\Users\nicol\AppData\Local\Temp\ipykernel_8852\2130195869.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["month"] = df["event_time"].dt.to_period("M")


In [29]:
df.groupby("month")["index"].nunique()

month
2019-10    407925
2019-11    462833
2019-12    351304
2020-01    443224
2020-02    429790
Freq: M, Name: index, dtype: int64

In [30]:
df["event_type"].value_counts(dropna=False)

event_type
view                965893
cart                585854
remove_from_cart    415754
purchase            127575
Name: count, dtype: int64

In [31]:
valid_events = {
    "view",
    "cart",
    "remove_from_cart",
    "purchase"
}

invalid_events = df.loc[
    ~df["event_type"].isin(valid_events),
    "event_type"
]

invalid_events.value_counts(dropna=False)

Series([], Name: count, dtype: int64)

In [32]:
df["price"].isna().sum()

np.int64(0)

In [33]:
(df["price"] < 0).sum()

np.int64(11)

In [34]:
(df["price"] == 0).sum()

np.int64(20533)

In [35]:
# 1. Diagnóstico: ¿cuántos registros tienen price negativo o cero?
neg = df[df["price"] < 0]
cero = df[df["price"] == 0]

print(f"Negativos: {len(neg)} ({len(neg)/len(df)*100:.3f}%)")
print(f"Cero: {len(cero)} ({len(cero)/len(df)*100:.3f}%)")

# 2. Mirá si esos negativos coinciden con algún patrón
# (ej: event_type == 'refund', 'return', 'cancel', etc.)
if "event_type" in df.columns:
    print(neg["event_type"].value_counts())

neg.head(10)

Negativos: 11 (0.001%)
Cero: 20533 (0.980%)
event_type
purchase    11
Name: count, dtype: int64


,index,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,month
143838,1426186,2019-10-10 14:33:29+00:00,purchase,5716855,1487580014042939619,NaN,NaN,-7.94,558797258,a406cf28-f04b-4361-8e6a-c62d36045e07,2019-10
1088665,2164651,2019-12-17 13:31:01+00:00,purchase,5670257,1487580014042939619,NaN,NaN,-15.87,513455730,efd60b1a-f616-4f48-a5d2-1a1ab0bd61e1,2019-12
1202594,3328682,2019-12-28 15:42:00+00:00,purchase,5716855,1487580014042939619,NaN,NaN,-7.94,539714006,e1d8e3b7-1309-475b-aaaa-4a12ac841c3e,2019-12
1210985,3418052,2019-12-29 17:49:32+00:00,purchase,5670257,1487580014042939619,NaN,NaN,-15.87,594077054,bf35396d-126d-4d6f-9f80-cd2eb0dd8a06,2019-12
1284494,595761,2020-01-06 15:31:18+00:00,purchase,5716857,1487580014042939619,NaN,NaN,-23.81,506281975,1263ab06-6a8a-408a-9a0b-e4e92dfdfa50,2020-01
1314973,890690,2020-01-08 20:58:08+00:00,purchase,5670257,1487580014042939619,NaN,NaN,-15.87,461649103,d99f464f-502b-4ce3-b289-634ba117991a,2020-01
1340194,1111694,2020-01-10 13:34:49+00:00,purchase,5716857,1487580014042939619,NaN,NaN,-23.81,598003172,d528824c-1308-4a52-af0e-bed078af64bb,2020-01
1398712,1673854,2020-01-14 10:53:59+00:00,purchase,5716857,1487580014042939619,NaN,NaN,-23.81,595587858,213f9bcd-f2a5-308d-6f3a-69182a7a18e1,2020-01
1401504,1702584,2020-01-14 13:46:58+00:00,purchase,5670257,1487580014042939619,NaN,NaN,-15.87,549622370,f236d393-6267-40a2-bad9-de3335ba4b53,2020-01
1519824,2834123,2020-01-22 11:02:29+00:00,purchase,5716857,1487580014042939619,NaN,NaN,-23.81,236833159,241ba301-17dd-488d-b464-e4ec1352ffff,2020-01


In [36]:
id_columns = [
    "product_id",
    "category_id",
    "user_id"
]

In [37]:
print("EVENT TYPES")
print(df["event_type"].value_counts(dropna=False))

print("\nINVALID EVENTS")
print(invalid_events.value_counts(dropna=False))

print("\nIDS")
print("Negative:")
print((df[id_columns] < 0).sum())

print("\nZero:")
print((df[id_columns] == 0).sum())

print("\nCardinality:")
print(df[id_columns].nunique())

EVENT TYPES
event_type
view                965893
cart                585854
remove_from_cart    415754
purchase            127575
Name: count, dtype: int64

INVALID EVENTS
Series([], Name: count, dtype: int64)

IDS
Negative:
product_id     0
category_id    0
user_id        0
dtype: int64

Zero:
product_id     0
category_id    0
user_id        0
dtype: int64

Cardinality:
product_id      46038
category_id       508
user_id        163936
dtype: int64


In [38]:
df[df["price"] < 0][
    ["event_time", "event_type", "product_id", "price", "user_id", "user_session"]
]

,event_time,event_type,product_id,price,user_id,user_session
143838,2019-10-10 14:33:29+00:00,purchase,5716855,-7.94,558797258,a406cf28-f04b-4361-8e6a-c62d36045e07
1088665,2019-12-17 13:31:01+00:00,purchase,5670257,-15.87,513455730,efd60b1a-f616-4f48-a5d2-1a1ab0bd61e1
1202594,2019-12-28 15:42:00+00:00,purchase,5716855,-7.94,539714006,e1d8e3b7-1309-475b-aaaa-4a12ac841c3e
1210985,2019-12-29 17:49:32+00:00,purchase,5670257,-15.87,594077054,bf35396d-126d-4d6f-9f80-cd2eb0dd8a06
1284494,2020-01-06 15:31:18+00:00,purchase,5716857,-23.81,506281975,1263ab06-6a8a-408a-9a0b-e4e92dfdfa50
1314973,2020-01-08 20:58:08+00:00,purchase,5670257,-15.87,461649103,d99f464f-502b-4ce3-b289-634ba117991a
1340194,2020-01-10 13:34:49+00:00,purchase,5716857,-23.81,598003172,d528824c-1308-4a52-af0e-bed078af64bb
1398712,2020-01-14 10:53:59+00:00,purchase,5716857,-23.81,595587858,213f9bcd-f2a5-308d-6f3a-69182a7a18e1
1401504,2020-01-14 13:46:58+00:00,purchase,5670257,-15.87,549622370,f236d393-6267-40a2-bad9-de3335ba4b53
1519824,2020-01-22 11:02:29+00:00,purchase,5716857,-23.81,236833159,241ba301-17dd-488d-b464-e4ec1352ffff


Se eliminaran los registros en la columna price que son negativos ya que no aportan a el analisis

In [39]:
df = df[df["price"] >= 0].copy()

In [40]:
df = df.drop(columns="index")

In [41]:
zero_price = df[df["price"] == 0]

zero_price.head(10)

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,month
343,2019-10-01 02:15:41+00:00,view,5892052,1487580010377117763,NaN,NaN,0.0,555455025,320f6021-30ac-4a58-ae17-bac1cc32aac3,2019-10
924,2019-10-01 05:16:30+00:00,view,5889621,1487580010561667147,NaN,NaN,0.0,523988665,00849bd2-fcd2-4cb4-af31-4e264f151848,2019-10
933,2019-10-01 05:18:03+00:00,view,5889622,1487580010561667147,NaN,NaN,0.0,523988665,80cfe614-f0a5-4101-a2b6-a21227590470,2019-10
937,2019-10-01 05:18:46+00:00,view,5889623,1487580010561667147,NaN,NaN,0.0,523988665,c2cd0464-3d2b-48e2-9667-bac248fe297a,2019-10
1077,2019-10-01 05:38:01+00:00,view,5889627,1487580010561667147,NaN,NaN,0.0,523988665,8b2bf9d8-43f0-43b2-bed3-13b2c956cada,2019-10
1079,2019-10-01 05:38:32+00:00,view,5889628,1487580010561667147,NaN,NaN,0.0,523988665,b7087089-41f5-48bc-b7eb-52fea86fa22c,2019-10
1082,2019-10-01 05:38:58+00:00,view,5889629,1487580010561667147,NaN,NaN,0.0,523988665,8d9aa6ed-53ca-4dcc-aa05-9c93de5fbbf8,2019-10
1084,2019-10-01 05:39:14+00:00,view,5889629,1487580010561667147,NaN,NaN,0.0,523988665,083ffacd-194b-48d6-b638-323632e28455,2019-10
1092,2019-10-01 05:40:40+00:00,view,5889631,1487580010561667147,NaN,NaN,0.0,523988665,cc622d89-5915-482b-b603-aa5e7f7d28c0,2019-10
1093,2019-10-01 05:40:54+00:00,view,5892179,1487580013950664926,NaN,NaN,0.0,465338762,36a90d24-5901-4424-adf0-fd84e0c53f34,2019-10


In [42]:
zero_price["event_type"].value_counts()

event_type
cart                10890
remove_from_cart     5312
view                 4331
Name: count, dtype: int64

In [43]:
zero_price["product_id"].nunique(), zero_price["product_id"].value_counts().head(20)

(7045,
 product_id
 5896186    79
 5903915    50
 5873428    37
 5851294    29
 5851304    29
 5837624    28
 5851272    27
 5712583    27
 5899512    26
 5907812    26
 5851295    25
 5851296    25
 5798000    25
 5851303    24
 5851263    24
 5798090    24
 5723523    24
 5772624    24
 5850652    24
 5851301    23
 Name: count, dtype: int64)

Los registros con price <= 0 fueron excluidos del dataset analítico debido a que no representan compras con valor monetario positivo. Los registros con price = 0 correspondían exclusivamente a eventos view, cart y remove_from_cart, sin eventos purchase. Esta decisión busca evitar distorsiones en análisis posteriores relacionados con precios, ingresos, AOV y LTV.

In [44]:
df = df[df["price"] > 0].copy()

In [45]:
df

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,month
0,2019-10-01 00:01:46+00:00,view,5843665,1487580005092295511,NaN,f.o.x,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361,2019-10
1,2019-10-01 00:01:55+00:00,cart,5868461,1487580013069861041,NaN,italwax,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1,2019-10
2,2019-10-01 00:02:50+00:00,view,5877456,1487580006300255120,NaN,jessnail,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770,2019-10
3,2019-10-01 00:03:41+00:00,view,5649270,1487580013749338323,NaN,concept,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df,2019-10
4,2019-10-01 00:03:44+00:00,view,18082,1487580005411062629,NaN,cnd,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5,2019-10
...,...,...,...,...,...,...,...,...,...,...
2095071,2020-02-29 23:58:49+00:00,cart,5815662,1487580006317032337,NaN,NaN,0.92,147995998,5ff96629-3627-493e-a25b-5a871ec78c90,2020-02
2095072,2020-02-29 23:58:57+00:00,view,5815665,1487580006317032337,NaN,NaN,0.59,147995998,5ff96629-3627-493e-a25b-5a871ec78c90,2020-02
2095073,2020-02-29 23:59:05+00:00,cart,5815665,1487580006317032337,NaN,NaN,0.59,147995998,5ff96629-3627-493e-a25b-5a871ec78c90,2020-02
2095074,2020-02-29 23:59:28+00:00,view,5817692,1487580010872045658,NaN,NaN,0.79,619841242,18af673b-7fb9-4202-a66d-5c855bc0fd2d,2020-02


In [46]:
df.info()

<class 'pandas.DataFrame'>
Index: 2074532 entries, 0 to 2095075
Data columns (total 10 columns):
 #   Column         Dtype              
---  ------         -----              
 0   event_time     datetime64[us, UTC]
 1   event_type     str                
 2   product_id     int64              
 3   category_id    int64              
 4   category_code  str                
 5   brand          str                
 6   price          float64            
 7   user_id        int64              
 8   user_session   str                
 9   month          period[M]          
dtypes: datetime64[us, UTC](1), float64(1), int64(3), period[M](1), str(4)
memory usage: 267.4 MB


In [47]:
df.shape

(2074532, 10)

In [49]:
processed_path = Path("../data/processed")

output_path = processed_path / "ecommerce_clean.parquet"

df.to_parquet(output_path, index=False)